# AutoShield AI: Supplier Risk Prediction Agent

## Project Overview

Automotive supply chains depend on the reliable delivery of critical components from geographically distributed suppliers.

Disruptions caused by delayed shipments, logistics bottlenecks, supplier constraints, and operational inefficiencies can significantly impact production schedules and revenue.

This notebook develops a machine learning-based Supplier Risk Prediction Agent capable of identifying potential supply chain disruptions before they occur.

The generated risk scores serve as the foundation for:

- Supplier Risk Monitoring
- Alternative Sourcing Recommendations
- Scenario Simulation
- Executive Decision Support

---

## Objectives

The primary objectives of this notebook are:

1. Build a supplier disruption prediction model.
2. Engineer risk-related supply chain features.
3. Train and evaluate a machine learning classifier.
4. Generate supplier-level disruption probabilities.
5. Export risk intelligence for downstream decision systems.

---

## Expected Outputs

- Trained Supplier Risk Model
- Disruption Probability Scores
- Supplier Risk Rankings
- Supplier Intelligence Dataset

# 1. Import Required Libraries

This section imports all libraries required for:

- Data Preparation
- Feature Engineering
- Machine Learning
- Model Evaluation
- Risk Scoring

The selected tools provide a robust workflow for developing predictive supply chain intelligence solutions.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

# 2. Load Digital Twin Dataset

The automotive digital twin dataset generated in the previous notebook is loaded as the primary data source for supplier risk modeling.

The dataset contains:

- Delivery Performance Metrics
- Supplier Information
- Geographic Attributes
- Commodity Categories
- Order Characteristics

These features collectively describe supply chain behavior and disruption patterns.

In [2]:
df = pd.read_csv(
    "../data/processed/automotive_digital_twin.csv",
    encoding="latin1"
)

print(df.shape)

(180519, 12)


# 3. Problem Formulation

## Business Objective

Predict whether a supplier transaction is likely to experience a delivery disruption.

## Target Variable

Late_delivery_risk

Interpretation:

- 1 → High Risk of Late Delivery
- 0 → Low Risk of Late Delivery

This prediction serves as an early warning signal for supply chain managers.

In [3]:
df["Late_delivery_risk"].value_counts()

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

# Class Distribution Analysis

Understanding class balance is important before model training.

A highly imbalanced dataset may require specialized handling techniques.

The observed distribution indicates that both classes are sufficiently represented, allowing standard supervised learning techniques to be applied effectively.

# 4. Feature Selection

Features are selected based on their potential relationship with delivery performance and supply chain disruption.

## Numerical Features

- Delay days
- Days for shipping (real)
- Days for shipment (scheduled)
- Order Item Quantity
- Sales

## Categorical Features

- Order Region
- Order Country
- Shipping Mode

These features capture operational, geographic, and logistics-related risk signals.

In [4]:
features = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delay_Days",
    "Order Item Quantity",
    "Sales",
    "Order Region",
    "Order Country",
    "Shipping Mode"
]

target = "Late_delivery_risk"

In [5]:
X = df[features]
y = df[target]

# 5. Train-Test Split

To evaluate model generalization performance, the dataset is divided into:

- Training Set (80%)
- Test Set (20%)

Stratified sampling is used to preserve the original class distribution.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(144415, 8)
(36104, 8)


# 6. Data Preprocessing Pipeline

The dataset contains both numerical and categorical features.

A preprocessing pipeline is constructed to:

- Preserve numerical features
- Encode categorical variables
- Ensure consistent transformations during inference

This approach improves maintainability and deployment readiness.

In [7]:
categorical_features = [
    "Order Region",
    "Order Country",
    "Shipping Mode"
]

numerical_features = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delay_Days",
    "Order Item Quantity",
    "Sales"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# 7. Model Development

An XGBoost Classifier is selected as the primary predictive model for supplier disruption risk.

## Why XGBoost?

XGBoost is one of the most widely adopted machine learning algorithms for structured and tabular business data due to its:

- High predictive performance
- Ability to capture complex nonlinear relationships
- Robust handling of mixed feature types
- Built-in regularization to reduce overfitting
- Strong performance in risk prediction and operational analytics

In supply chain environments, XGBoost is frequently used for:

- Disruption Prediction
- Demand Forecasting
- Inventory Optimization
- Logistics Risk Assessment

The model is trained to predict the probability of late deliveries, which serves as a proxy for supplier disruption risk.

In [8]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

In [9]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [10]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# 8. Model Evaluation

The trained model is evaluated using:

## Accuracy

Measures overall prediction correctness.

## ROC-AUC

Measures the model's ability to distinguish between high-risk and low-risk deliveries.

## Precision

Measures reliability of positive predictions.

## Recall

Measures ability to detect disruptions.

Together these metrics provide a comprehensive view of model effectiveness.

In [11]:
preds = pipeline.predict(X_test)

probs = pipeline.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, preds))

print("ROC-AUC:", roc_auc_score(y_test, probs))

print(classification_report(y_test, preds))

Accuracy: 0.9744626634167959
ROC-AUC: 0.9757025275668082
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     16308
           1       0.96      1.00      0.98     19796

    accuracy                           0.97     36104
   macro avg       0.98      0.97      0.97     36104
weighted avg       0.98      0.97      0.97     36104



# Model Performance Summary

Results:

- Accuracy: 97.4%
- ROC-AUC: 97.6%

The model demonstrates excellent predictive capability and is highly effective at identifying potential delivery disruptions.

This performance supports the use of predicted probabilities as supplier risk indicators within the AutoShield AI platform.

# 9. Risk Score Generation

Rather than using only binary predictions, disruption probabilities are generated.

These probabilities provide a continuous measure of supplier risk and allow:

- Supplier Ranking
- Risk Segmentation
- Alternative Sourcing Prioritization
- Executive Risk Monitoring

In [12]:
df["Risk_Probability"] = (
    pipeline.predict_proba(X)
    [:,1]
)

In [13]:
df["Risk_Probability"].describe()

count    180519.000000
mean          0.548539
std           0.473818
min           0.000005
25%           0.000016
50%           0.948034
75%           0.958313
max           0.996763
Name: Risk_Probability, dtype: float64

# Business Interpretation

Higher disruption probabilities indicate suppliers that may require:

- Increased monitoring
- Inventory buffering
- Contingency planning
- Alternative sourcing strategies

These risk scores become a critical input for supply chain resilience planning.

# 10. Export Risk Intelligence Dataset

The generated disruption probabilities are exported for downstream modules:

- Supplier Intelligence Engine
- Alternative Sourcing Engine
- Scenario Simulator
- Executive AI Copilot

This enables consistent risk-driven decision making across the platform.

In [14]:
df.to_csv(
    "../data/risk_scored_supply_chain.csv",
    index=False
)

# Conclusion

This notebook developed the Supplier Risk Prediction Agent for AutoShield AI.

Key accomplishments include:

- Supply chain risk feature engineering
- Machine learning-based disruption prediction
- High-performance risk classification
- Supplier-level probability scoring

The generated risk intelligence serves as the foundation for proactive supply chain resilience and decision support capabilities within the AutoShield AI platform.